# Neden OCR ?

Bir önceki hizalama methodumuzda ham ses sinyallerini işledik. Fakat bu yöntem bazı durumlarda kararsızlaşabilir veya sağlıklı çalışmayabilir. Fakat maç saati genellikle yayına gömülü bir şekilde gelir ve ses değişse ve görüntüde sabit kalır.

Ayrıca Clock direkt olarak maçın kendi iç zaman çizgisini doğrudan verir. Ekranda 67:23 okunduğunda bu zaten maçın hangi anı olduğunu söyleyen semantik bir zaman damgasıdır.

Full maç videsounda ve özet videosunda aynı grafikler kullanıldığı için OCR methodunun yaptığı hatalar bile tutarlı olacaktır.

Burada önemli olan ROI diye isimlendirdiğimiz skor tablosu grafiğinin görüntüde nerede olduğudur. Bu sayede metnin nerede olduğu (detection) atlanır, ve direkt olarak metnin okunması aşamasına (recognition) geçilir.

Bu katmandan sonra da full maçın zamanı ekseninde 0-1 etiketi üreteceğiz. Özette görünen her clock değeri, full maçtaki karşılığınıı eşlenip o saniyeye 1 geri kalanlara 0 diyeceğiz.

# Video --> Kare

OCR, videoların tamamını değil, saniyede sadece 1 kareyi işler. Çünkü:

90 dakikalık bir maçta 25 fps'te yaklaşık olarak 135.000 kare eder. Bunu saniyede 1 kareye örnekleyince yaklaşık olarak 5.400'e düşer. E zaten maç saati saniyede 1 değişeceği için her saniyede 25 kere örneklemek verimsizdir. Böylece işlem süresini kat kat düşürebiliriz.



# ROI (Region of Interest)

ROI yani ilgilendiğimiz bölge, karenin yalnızca işimize yarayan kısmıdır, yani maç saati. Her bir frame de bu bölgeyi keseceğiz. Çünkü skor tablosu her maçta (UEFA tarafından gerçekleştirilen müsabakalar için) aynı yerde olduğu için maç saati metninin nerede olduğunu aramamıza gerek yoktur.

Burada ROI'yi olabildiğince dar tutmak bizim için daha iyi, çünkü maç saatini okuma sırasında skorboard gibi sayıları da okuma ihtimalimiz çok yüksek.

ROI koordinatlarını piksel cinsinden değil, frame'in genişlik ve yüksekliğine oranla 0-1 arasında bir sayı ile tanımlarız. Örneğin kabaca:

yatay da x ∈ [0.04, 0.13]

dikeyde y ∈ [0.05, 0.12]

aralığındadır. Peki bunu neden piksel yerine oran şeklinde aldık ? Çünkü kullandığımız maçların yayın çözünürlükleri farklı olabilir. 720p, 1080p gibi. Piksek koordinatları her çözünürlük için farklı olacağı için oran daha robusttur.

Tek sorun ROI biraz yayıncıya bağlıdır. Farklı yayıncı kuruluşların yayınları için bu ROI güncellenmesi gerekebilir.


# Preprocessing

Yukarıda bahsettiğimiz ROI'yi kırptıktan sonra saat görüntüsü OCR moturuna olduğu gibi verilebilir, fakat pek verimli çalışmaz. Bu aşamada motorun saat bilgisini okuması için kırptığımız bölge üzerinde çalışacağız.

1. Upscale - büyütme:

Saatin olduğu bölge düşük çözünürlükte olacağı ve OCR motorları büyük yazıları daha iyi okuyabildiği için kırptığımız bölgeyi birkaç kat büyütürüz. Bu işlem bilgi v.s. eklemez, sadece motorun sayıları daha doğru bir şekilde okuması için yapılır.

2. Grayscale - Gri tonlama:

Maç saatinin rakamları için renk hiç önemli değildir. Renk kanallarını tek bir parlaklık kanalına indirmek hem gereksiz bilgiyi atar hem de sondaki eşikleme işlemini basitleştirir. Bu şekilde daha verimli olacaktır.

3. Threshold - eşikleme:

Maç saatinin grafiğinin tasarımı koyu bir zemin üzerinde beyaz rakamlar olduğu için bu yüksek kontrast basit bir eşikleme ile rakamları arka plandan kolayca ayırabilir.


Bu aşamadan sonra yalnızca beyaz saat rakamlarının net bir şekilde okunduğu bir görüntü elde edeceğiz, her frame için. Bu aşamadaki doğrulamalar gözle test edildi.


# OCR Motoru
Bu aşamada işlenmiş maç saati görüntüsünü bir string'e çevirmeye çalışacağız. Bunun için bir kaç yol denedik, fakat en düzgün çalışanı PaddleOCR oldu. (Tesseract, template matching v.s.)

PaddleOCR kullanırken belge yönü sınıflandırma, belge açma v.s. gibi özelliklerini kullanmadığımız için kapattık.

Ayrıca OCR motorunun ürettiği stringleri filtrelemek için regex kullandık. Böylece bu motorun yanlış veya gereksiz okuduğu | gibi çöp simgelerden kurtulduk.


Bu aşamanın sonunda "09:00" gibi temiz bir string çıktı elde etmeyi bekledik.

# Sahne Tespiti

Bu aşamaya kadar bir frame alarak o framede'ki saati okuduk. Fakat bu methodu hangi frame'lere uygulayacağımıza karar vermedik. Özet videoyu sabit aralıklarla örneklemek yetersiz kaldı. Bizim istediğimiz her sahnenin başlangıcı ve bitişini tespit etmek.

Bunun için PySceneDetect modülünün ContentDetector kullandık. Bu modülün temel fikri iki frame'in içerik farkını ölçerek sahne değişimini tespit eder.

Yazdığımız ses hizalama methodu 12 sahne bulurken bu method aynı maç içerisinde 25-30 pozisyon bulabildi. Böylece OCR methodu audio fingerprint'in kaçırdıklarını da yakalayabildi.

Her sahnenin başlangıç ve bitişini yakaladıktan sonra tam orta frame'i alınıp maaç saati okuma işlemini gerçekleştirdik.

Bu aşama sonunda özet, sahnelere bölünmüş ve her sahne için bir maç saati (eğer ki replay v.s. denk gelip maç saatini bulamazsa None) veya None'a eşlenmiş olur.

# Correspondence - Yakaladığımız sahneleri Full Maça Eşleme

Bu bölümde yapmak istediğimiz şey, özet videosunda yakalayıp maç saatini okuduğumuz sahnelerin, full maç videosunda nereye denk geldiğidir. Burada ortak eksenimiz maç saatidir.

Fakat full maçta maç saati ile video saati arasında fark vardır. (maç önü, maç sonu, devre arası v.s.) Buna kodda offsett dedik. Bu offsett her maç için farklı olacaktır. Bu sebeple ilk olarak offsetti bulmamız gerekir.

Burada bir önceki yazdığımız yol olan audio fingerprintten yardım alacağız. Fingerprint bir sahnenin video-saniyesini bize vermekte. (seslerin eşleşmesinden). OCR ise aynı sahnenin bize maç saati vermekte. Burada ikisininde aynı sahneyi tespit ettiğini bulursak, farkları doğrudan offsetti verecektir.

Örneğin, Fingerprint bize bir sahne için 3885.4 sn verdi, OCR ise 60:25 de bir sahne tespit etti (3625 sn)
Burada offsetimiz 260.4. Tek bir offsete güvenmemek için maçın ikinci yarısından alınan birden çok sahnelerin offsetleri hesaplanır ve medyanları alınır. Medyan sayesinde outlier değerler elenir.

Şimdi elimizde offsett değeri olunca özetten gelen her sahne için full maç videosunda hangi video saniyesine geldiğini bulabiliriz.





# Füzyon - İki Hizalamayı Birleştirme

Bu noktaya kadar 2 bağımsız yol ile  full maç videosu uzunluğunda ve 0-1 değerlerden oluşan bir vektör olusturduk. Bu vektörler sayesinde modele videonun hangi anın önemli olduğunu verebiliriz. Peki bu 2 ayrı methottan hangisine güveneceğiz?

Bu füzyon katmanında bu iki yolu nihai tek bir etikete birleştirdik. İki yolda birbirinin tamamlayıcısı niteliğinde. Dolayısıyla hangisi daha iyi seçmek değil, ikisinin yakaladıklarını toplamak.

Bu birleştirme işlemini mantıksal veya işlemi ile yaptık. bir saniye için, katmanlardan herhangi biri onu 1 işaretlediyse nihai etikette 1 olur.


Fakat bu birleşim çok keskin bir şekilde yaptık. Yani bir saniye veya bir an, ya önemli ya önemsiz gibi oldu. Model için bu çok sert bir geçiş. Yani bu daha kademeli yapmalıyız. Ayrıca birkaç saniye kaymaları da bu şekilde telafi edebiliriz. Bu yüzden gaussian_filter1d uyguladık.
Bu method sayesinde her 1 bloğunu etrafına yayılan yumuşak bir tepeye dönüştürür: merkez en yüksek, kenarlara doğru kademeli azalan bir önem eğrisi. Sonuç, 0 ile 1 arasında sürekli bir değer alan bir skor vektörüdü.



# Sonuc

Bu ürettiğimiz sürekli skor vektörünü numpy array olarak kaydettik. Böylece eğitim için hazır hale geldi.

Füzyonun sonunda, full maçın her saniyesi için [0,1] aralığında sürekli bir önem skoru elde ettik. Bu, hizalama pipeline'ının nihai ürünüdür — parmak izi ve OCR'ın birlikte ürettiği, modelin öğreneceği eğitim hedefi.